# Module 7 — MAML-DSCNN-SE-DA-3C
## EEGMMIDB (109 subjects) → BCI-IV-2a (9 subjects)

**Goal:** the final cross-dataset experiment for the prepared 3-class EEG cache.

### Fixed interface
- EEG input: `(N, 22, 640)`
- Sampling rate: 160 Hz
- Classes: `left`, `right`, `feet`
- Source subjects: 109 EEGMMIDB subjects
- External target subjects: 9 BCI-IV-2a subjects

### Architecture
1. Multi-scale temporal depthwise-separable CNN (`k=15,31,63`)
2. Depthwise spatial filtering across all 22 electrodes
3. Squeeze-and-Excitation (SE) channel attention
4. 128-D subject-invariant embedding
5. First-order MAML-style episodic source training across EEGMMIDB subjects
6. Gradient-reversal subject-domain alignment during source fine-tuning
7. 3-class classifier

### Evaluation
**Strict:** BCI-IV-2a is never used for training, normalization, validation, or model selection.

**Transductive:** the BCI target EEG values may be used to compute target-only channel normalization; BCI labels remain unused until final scoring.

The published MAML-LNN paper reports 81.3% cross-subject accuracy on BCI-IV-2a in a different four-class experimental protocol. EEG-TriNet++ reports 72.3% BCI-IV-2a and 70.8% PhysioNet LOSO. Those numbers are benchmarks, not guarantees on this cache.

In [1]:
# ============================================================
# CELL 1 — IMPORTS / GLOBAL CONFIGURATION
# ============================================================

from __future__ import annotations

import os
import gc
import copy
import time
import json
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import h5py

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

from torch.utils.data import (
    TensorDataset,
    DataLoader,
    WeightedRandomSampler,
)

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    cohen_kappa_score,
    confusion_matrix,
    classification_report,
)

warnings.filterwarnings("ignore")

SEED = 42
SEEDS = (42, 123)

# ------------------------------------------------------------
# Fast execution switches
# ------------------------------------------------------------

SMOKE_MODE = True

# Full source training settings.
FULL_META_EPOCHS = 20
FULL_EPISODES_PER_EPOCH = 15
FULL_TASKS_PER_EPISODE = 4
FULL_SUPPORT_PER_TASK = 32
FULL_QUERY_PER_TASK = 32
FULL_FINAL_EPOCHS = 60
FULL_BATCH_SIZE = 64

# Smoke settings.
SMOKE_META_EPOCHS = 3
SMOKE_EPISODES_PER_EPOCH = 5
SMOKE_TASKS_PER_EPISODE = 3
SMOKE_SUPPORT_PER_TASK = 24
SMOKE_QUERY_PER_TASK = 24
SMOKE_FINAL_EPOCHS = 15
SMOKE_BATCH_SIZE = 64

INNER_LR = 1e-3
META_LR = 2e-4
FINAL_LR = 7e-4
WEIGHT_DECAY = 2e-4

DOMAIN_LAMBDA = 0.035
DOMAIN_LR = 5e-4

VAL_SUBJECT_FRACTION = 0.20

DEVICE = (
    torch.device("cuda")
    if torch.cuda.is_available()
    else torch.device("cpu")
)


def seed_everything(seed=42):

    os.environ["PYTHONHASHSEED"] = str(seed)

    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)

    if torch.cuda.is_available():

        torch.cuda.manual_seed_all(seed)

        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False


seed_everything(SEED)

print("Device:", DEVICE)
print("SMOKE_MODE:", SMOKE_MODE)

Device: cpu
SMOKE_MODE: True


In [2]:
# ============================================================
# CELL 2 — EXISTING PROJECT CACHE
# ============================================================

PROJECT_ROOT = Path(
    "/Users/ashokvarmabevara/Project2"
)

PROJECT_DIR = (
    PROJECT_ROOT
    / "cross_dataset_mi_project"
)

CACHE_DIR = (
    PROJECT_DIR
    / "cache"
)

RESULT_DIR = (
    PROJECT_DIR
    / "results"
    / "module_7_maml_dscnn_se_da"
)

RESULT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

EXPECTED_CACHE = (
    CACHE_DIR
    / "module_5_v2_preprocessed_epochs_160hz_8_30hz_continuous.h5"
)

if EXPECTED_CACHE.exists():

    CACHE_PATH = EXPECTED_CACHE

else:

    candidates = sorted(
        CACHE_DIR.glob("*.h5")
    )

    preferred = [
        p for p in candidates
        if (
            "160hz" in p.name.lower()
            or "module_5" in p.name.lower()
        )
    ]

    if not preferred:

        raise FileNotFoundError(
            f"No .h5 cache found in {CACHE_DIR}"
        )

    CACHE_PATH = preferred[0]

print(
    "Cache:",
    CACHE_PATH,
)

assert CACHE_PATH.exists()

Cache: /Users/ashokvarmabevara/Project2/cross_dataset_mi_project/cache/module_5_v2_preprocessed_epochs_160hz_8_30hz_continuous.h5


In [3]:
# ============================================================
# CELL 3 — READ METADATA / VERIFY 109 + 9 = 118
# ============================================================

def decode(v):

    return (
        v.decode("utf-8")
        if isinstance(v, bytes)
        else str(v)
    )


with h5py.File(
    CACHE_PATH,
    "r",
) as h5:

    X_SHAPE = tuple(
        h5["X"].shape
    )

    X_DTYPE = str(
        h5["X"].dtype
    )

    metadata = {}

    for key in [
        "dataset",
        "subject",
        "run",
        "recording_id",
        "filename",
        "absolute_path",
        "harmonized_class",
    ]:

        metadata[key] = [
            decode(v)
            for v in h5[
                "metadata"
            ][key][:]
        ]


META = pd.DataFrame(
    metadata
)

META.insert(
    0,
    "cache_index",
    np.arange(
        len(META),
        dtype=np.int64,
    ),
)

CLASSES = [
    "left",
    "right",
    "feet",
]

CLASS_TO_ID = {
    c: i
    for i, c in enumerate(CLASSES)
}

ID_TO_CLASS = {
    i: c
    for c, i in CLASS_TO_ID.items()
}

N_CLASSES = 3

assert X_SHAPE[1:] == (
    22,
    640,
)

assert X_DTYPE == "float32"

EEGMMIDB_META = META[
    META[
        "dataset"
    ]
    .astype(str)
    .str.upper()
    .eq("EEGMMIDB")
].copy()

BCI_META = META[
    META[
        "dataset"
    ]
    .astype(str)
    .eq("BCI-IV-2a")
].copy()

EEGMMIDB_META["subject"] = (
    EEGMMIDB_META[
        "subject"
    ].astype(str)
)

BCI_META["subject"] = (
    BCI_META[
        "subject"
    ].astype(str)
)

SOURCE_SUBJECTS = sorted(
    EEGMMIDB_META[
        "subject"
    ].unique()
)

TARGET_SUBJECTS = sorted(
    BCI_META[
        "subject"
    ].unique()
)

print(
    "Cache shape:",
    X_SHAPE,
)

print(
    "EEGMMIDB subjects:",
    len(SOURCE_SUBJECTS),
)

print(
    "BCI-IV-2a subjects:",
    len(TARGET_SUBJECTS),
)

print(
    "Total subjects:",
    len(SOURCE_SUBJECTS)
    + len(TARGET_SUBJECTS),
)

assert len(
    SOURCE_SUBJECTS
) == 109

assert len(
    TARGET_SUBJECTS
) == 9

Cache shape: (9316, 22, 640)
EEGMMIDB subjects: 109
BCI-IV-2a subjects: 9
Total subjects: 118


In [4]:
# ============================================================
# CELL 4 — HDF5 LOADER / NUMERICAL QA
# ============================================================

def load_indices(
    indices,
):

    indices = np.asarray(
        indices,
        dtype=np.int64,
    )

    with h5py.File(
        CACHE_PATH,
        "r",
    ) as h5:

        return np.asarray(
            h5["X"][indices],
            dtype=np.float32,
        )


qa_idx = np.arange(
    min(
        512,
        X_SHAPE[0],
    ),
    dtype=np.int64,
)

X_QA = load_indices(
    qa_idx
)

nonfinite = int(
    (
        ~np.isfinite(
            X_QA
        )
    ).sum()
)

zero_variance = int(
    (
        np.var(
            X_QA,
            axis=(1,2),
        )
        <= 1e-12
    ).sum()
)

print(
    "Non-finite:",
    nonfinite,
)

print(
    "Zero variance:",
    zero_variance,
)

assert nonfinite == 0

print(
    "✅ Numerical QA PASS"
)

Non-finite: 0
Zero variance: 0
✅ Numerical QA PASS


In [5]:
# ============================================================
# CELL 5 — ROBUST NORMALIZATION
# ============================================================

class RobustChannelNormalizer:

    def __init__(
        self,
        eps=1e-6,
    ):

        self.eps = eps
        self.center_ = None
        self.scale_ = None

    def fit(
        self,
        X,
    ):

        X = np.asarray(
            X,
            dtype=np.float32,
        )

        V = (
            X
            .transpose(
                1,
                0,
                2,
            )
            .reshape(
                X.shape[1],
                -1,
            )
        )

        self.center_ = np.median(
            V,
            axis=1,
        )

        q25 = np.percentile(
            V,
            25,
            axis=1,
        )

        q75 = np.percentile(
            V,
            75,
            axis=1,
        )

        self.scale_ = np.maximum(
            q75 - q25,
            self.eps,
        )

        return self

    def transform(
        self,
        X,
    ):

        if self.center_ is None:

            raise RuntimeError(
                "Normalizer is not fitted."
            )

        X = np.asarray(
            X,
            dtype=np.float32,
        )

        Z = (
            X
            - self.center_[
                None,
                :,
                None,
            ]
        )

        Z = (
            Z
            / (
                self.scale_[
                    None,
                    :,
                    None,
                ]
                + self.eps
            )
        )

        Z = np.nan_to_num(
            Z,
            nan=0.0,
            posinf=0.0,
            neginf=0.0,
        )

        return Z.astype(
            np.float32
        )


def target_subject_normalize(
    X_subject,
):

    norm = (
        RobustChannelNormalizer()
        .fit(
            X_subject
        )
    )

    return (
        norm.transform(
            X_subject
        ),
        norm,
    )

In [6]:
# ============================================================
# CELL 6 — GROUPED SOURCE TRAIN/VALIDATION SPLIT
# ============================================================

def grouped_source_split(
    subjects,
    fraction=0.20,
    seed=42,
):

    subjects = list(
        map(
            str,
            subjects,
        )
    )

    rng = np.random.default_rng(
        seed
    )

    shuffled = subjects.copy()

    rng.shuffle(
        shuffled
    )

    n_val = max(
        1,
        int(
            round(
                len(shuffled)
                * fraction
            )
        ),
    )

    val_subjects = sorted(
        shuffled[
            :n_val
        ]
    )

    train_subjects = sorted(
        shuffled[
            n_val:
        ]
    )

    return (
        train_subjects,
        val_subjects,
    )


TRAIN_SUBJECTS, VAL_SUBJECTS = (
    grouped_source_split(
        SOURCE_SUBJECTS,
        fraction=VAL_SUBJECT_FRACTION,
        seed=SEED,
    )
)

print(
    "Source train subjects:",
    len(TRAIN_SUBJECTS),
)

print(
    "Source validation subjects:",
    len(VAL_SUBJECTS),
)

print(
    "Validation example:",
    VAL_SUBJECTS[:15],
)

assert (
    len(
        set(TRAIN_SUBJECTS)
        &
        set(VAL_SUBJECTS)
    )
    == 0
)

assert (
    len(TRAIN_SUBJECTS)
    +
    len(VAL_SUBJECTS)
    ==
    len(SOURCE_SUBJECTS)
)

Source train subjects: 87
Source validation subjects: 22
Validation example: ['S001', 'S003', 'S005', 'S019', 'S025', 'S029', 'S034', 'S038', 'S047', 'S060', 'S069', 'S071', 'S073', 'S079', 'S080']


In [7]:
# ============================================================
# CELL 7 — LOAD SOURCE TRAIN / SOURCE VAL
# ============================================================

def get_arrays_by_subjects(
    meta_df,
    subjects,
):

    subjects = set(
        str(s)
        for s in subjects
    )

    mask = (
        meta_df[
            "subject"
        ]
        .astype(str)
        .isin(subjects)
    )

    idx = (
        meta_df.loc[
            mask,
            "cache_index",
        ]
        .to_numpy(
            dtype=np.int64
        )
    )

    X = load_indices(
        idx
    )

    y = (
        meta_df.loc[
            mask,
            "harmonized_class",
        ]
        .map(
            CLASS_TO_ID
        )
        .to_numpy(
            dtype=np.int64
        )
    )

    subjects_arr = (
        meta_df.loc[
            mask,
            "subject",
        ]
        .astype(str)
        .to_numpy()
    )

    return (
        X,
        y,
        subjects_arr,
        idx,
    )


X_SOURCE_TRAIN_RAW, y_source_train, sub_source_train, idx_source_train = (
    get_arrays_by_subjects(
        EEGMMIDB_META,
        TRAIN_SUBJECTS,
    )
)

X_SOURCE_VAL_RAW, y_source_val, sub_source_val, idx_source_val = (
    get_arrays_by_subjects(
        EEGMMIDB_META,
        VAL_SUBJECTS,
    )
)

print(
    "Train:",
    X_SOURCE_TRAIN_RAW.shape,
)

print(
    "Val  :",
    X_SOURCE_VAL_RAW.shape,
)

print(
    "Train class counts:",
    np.bincount(
        y_source_train,
        minlength=3,
    ),
)

print(
    "Val class counts:",
    np.bincount(
        y_source_val,
        minlength=3,
    )
)

Train: (5886, 22, 640)
Val  : (1486, 22, 640)
Train class counts: [1979 1948 1959]
Val class counts: [500 490 496]


In [8]:
# ============================================================
# CELL 8 — PREPARE SOURCE NORMALIZATION
# ============================================================

SOURCE_NORM = (
    RobustChannelNormalizer()
    .fit(
        X_SOURCE_TRAIN_RAW
    )
)

X_SOURCE_TRAIN = (
    SOURCE_NORM.transform(
        X_SOURCE_TRAIN_RAW
    )
)

X_SOURCE_VAL = (
    SOURCE_NORM.transform(
        X_SOURCE_VAL_RAW
    )
)

print(
    "Normalized train:",
    X_SOURCE_TRAIN.shape,
)

print(
    "Normalized val:",
    X_SOURCE_VAL.shape,
)

print(
    "✅ Source-only normalization ready."
)

Normalized train: (5886, 22, 640)
Normalized val: (1486, 22, 640)
✅ Source-only normalization ready.


In [9]:
# ============================================================
# CELL 9 — MULTI-SCALE DSCNN + SE ENCODER
# ============================================================

class SEBlock(
    nn.Module
):

    def __init__(
        self,
        channels,
        reduction=8,
    ):

        super().__init__()

        hidden = max(
            8,
            channels // reduction,
        )

        self.fc1 = nn.Conv1d(
            channels,
            hidden,
            kernel_size=1,
        )

        self.fc2 = nn.Conv1d(
            hidden,
            channels,
            kernel_size=1,
        )

    def forward(
        self,
        x,
    ):

        # x = B,C,T
        s = x.mean(
            dim=2,
            keepdim=True,
        )

        s = F.relu(
            self.fc1(s)
        )

        s = torch.sigmoid(
            self.fc2(s)
        )

        return x * s


class TemporalSpatialBranch(
    nn.Module
):

    def __init__(
        self,
        n_channels=22,
        out_channels=32,
        kernel_size=31,
        dropout=0.20,
    ):

        super().__init__()

        # Temporal convolution.
        self.temporal = nn.Conv2d(
            1,
            16,
            kernel_size=(
                1,
                kernel_size,
            ),
            padding=(
                0,
                kernel_size // 2,
            ),
            bias=False,
        )

        self.gn1 = nn.GroupNorm(
            4,
            16,
        )

        # Spatial depthwise filtering.
        self.spatial = nn.Conv2d(
            16,
            out_channels,
            kernel_size=(
                n_channels,
                1,
            ),
            groups=16,
            bias=False,
        )

        self.gn2 = nn.GroupNorm(
            8,
            out_channels,
        )

        # Separable temporal refinement.
        self.depth = nn.Conv2d(
            out_channels,
            out_channels,
            kernel_size=(
                1,
                15,
            ),
            padding=(
                0,
                7,
            ),
            groups=out_channels,
            bias=False,
        )

        self.point = nn.Conv2d(
            out_channels,
            out_channels,
            kernel_size=1,
            bias=False,
        )

        self.gn3 = nn.GroupNorm(
            8,
            out_channels,
        )

        self.pool = nn.AvgPool2d(
            kernel_size=(
                1,
                4,
            ),
            stride=(
                1,
                4,
            ),
        )

        self.dropout = nn.Dropout(
            dropout
        )

    def forward(
        self,
        x,
    ):

        z = x.unsqueeze(
            1
        )

        z = self.temporal(
            z
        )

        z = F.gelu(
            self.gn1(z)
        )

        z = self.spatial(
            z
        )

        z = F.gelu(
            self.gn2(z)
        )

        z = self.depth(
            z
        )

        z = self.point(
            z
        )

        z = F.gelu(
            self.gn3(z)
        )

        z = self.pool(
            z
        )

        z = self.dropout(
            z
        )

        # B,C,1,T
        z = z.squeeze(
            2
        )

        return z


class DSCNNSEEncoder(
    nn.Module
):

    def __init__(
        self,
        n_channels=22,
        embedding_dim=128,
    ):

        super().__init__()

        branch_channels = 32

        self.branches = nn.ModuleList(
            [
                TemporalSpatialBranch(
                    n_channels=n_channels,
                    out_channels=branch_channels,
                    kernel_size=15,
                ),

                TemporalSpatialBranch(
                    n_channels=n_channels,
                    out_channels=branch_channels,
                    kernel_size=31,
                ),

                TemporalSpatialBranch(
                    n_channels=n_channels,
                    out_channels=branch_channels,
                    kernel_size=63,
                ),
            ]
        )

        fused_channels = (
            branch_channels * 3
        )

        self.se = SEBlock(
            fused_channels,
            reduction=8,
        )

        self.fusion = nn.Sequential(

            nn.Conv1d(
                fused_channels,
                embedding_dim,
                kernel_size=1,
                bias=False,
            ),

            nn.GroupNorm(
                8,
                embedding_dim,
            ),

            nn.GELU(),

            nn.Dropout(
                0.20
            ),
        )

        # Attentive temporal pooling.
        self.temporal_score = nn.Sequential(

            nn.Conv1d(
                embedding_dim,
                32,
                kernel_size=1,
            ),

            nn.GELU(),

            nn.Conv1d(
                32,
                1,
                kernel_size=1,
            ),
        )

    def forward(
        self,
        x,
    ):

        branches = [
            branch(x)
            for branch in self.branches
        ]

        z = torch.cat(
            branches,
            dim=1,
        )

        z = self.se(
            z
        )

        z = self.fusion(
            z
        )

        scores = (
            self.temporal_score(
                z
            )
        )

        weights = torch.softmax(
            scores,
            dim=2,
        )

        embedding = (
            z
            * weights
        ).sum(
            dim=2
        )

        return embedding


class MAMLDSCNNSE3C(
    nn.Module
):

    def __init__(
        self,
        n_channels=22,
        embedding_dim=128,
        n_classes=3,
        n_domains=109,
    ):

        super().__init__()

        self.encoder = (
            DSCNNSEEncoder(
                n_channels=n_channels,
                embedding_dim=embedding_dim,
            )
        )

        self.classifier = nn.Sequential(

            nn.Linear(
                embedding_dim,
                64,
            ),

            nn.GELU(),

            nn.Dropout(
                0.25
            ),

            nn.Linear(
                64,
                n_classes,
            ),
        )

        self.domain_classifier = nn.Sequential(

            nn.Linear(
                embedding_dim,
                64,
            ),

            nn.GELU(),

            nn.Dropout(
                0.15
            ),

            nn.Linear(
                64,
                n_domains,
            ),
        )

    def forward(
        self,
        x,
        return_features=False,
    ):

        z = self.encoder(
            x
        )

        logits = self.classifier(
            z
        )

        if return_features:

            return (
                logits,
                z,
            )

        return logits

    def domain_logits(
        self,
        features,
    ):

        return self.domain_classifier(
            features
        )


# ------------------------------------------------------------
# Forward test
# ------------------------------------------------------------

test_model = (
    MAMLDSCNNSE3C().to(
        DEVICE
    )
)

test_model.eval()

with torch.no_grad():

    dummy = torch.randn(
        4,
        22,
        640,
        device=DEVICE,
    )

    dummy_logits, dummy_features = (
        test_model(
            dummy,
            return_features=True,
        )
    )

print(
    "Logits:",
    tuple(
        dummy_logits.shape
    ),
)

print(
    "Embedding:",
    tuple(
        dummy_features.shape
    ),
)

print(
    "Parameters:",
    f"{sum(p.numel() for p in test_model.parameters()):,}",
)

assert tuple(
    dummy_logits.shape
) == (
    4,
    3,
)

assert tuple(
    dummy_features.shape
) == (
    4,
    128,
)

del test_model
gc.collect()

print(
    "✅ MAML-DSCNN-SE-3C forward PASS."
)

Logits: (4, 3)
Embedding: (4, 128)
Parameters: 51,757
✅ MAML-DSCNN-SE-3C forward PASS.


In [10]:

# ============================================================
# CELL 10 — MAML TASK SAMPLING
# ============================================================

def subject_to_indices(
    subject_array,
):

    mapping = {}

    for subject in np.unique(
        subject_array
    ):

        mapping[
            str(subject)
        ] = np.flatnonzero(
            subject_array
            == subject
        )

    return mapping


TRAIN_SUBJECT_INDEX_MAP = (
    subject_to_indices(
        sub_source_train
    )
)


def sample_task(
    X,
    y,
    subject_array,
    rng,
    support_size=32,
    query_size=32,
):

    subjects = list(
        TRAIN_SUBJECT_INDEX_MAP.keys()
    )

    subject = rng.choice(
        subjects
    )

    # Re-index from the global source-train arrays.
    idx = (
        TRAIN_SUBJECT_INDEX_MAP[
            subject
        ]
    )

    if len(idx) < (
        support_size
        + query_size
    ):

        replace = True

    else:

        replace = False

    selected = rng.choice(
        idx,
        size=(
            support_size
            + query_size
        ),
        replace=replace,
    )

    support_idx = selected[
        :support_size
    ]

    query_idx = selected[
        support_size:
    ]

    return (
        X[support_idx],
        y[support_idx],
        X[query_idx],
        y[query_idx],
    )

In [11]:
# ============================================================
# CELL 11 — DIFFERENTIABLE FUNCTIONAL CALL HELPER
# ============================================================

try:

    from torch.func import (
        functional_call
    )

    print(
        "Using torch.func.functional_call"
    )

except ImportError:

    from torch.nn.utils.stateless import (
        functional_call
    )

    print(
        "Using legacy functional_call"
    )


def functional_model_call(
    model,
    params,
    x,
):

    return functional_call(
        model,
        params,
        (
            x,
        ),
    )

Using torch.func.functional_call


In [12]:
# ============================================================
# CELL 12 — FIRST-ORDER MAML META-TRAINING
# ============================================================

def meta_train_one_seed(
    seed,
    epochs,
    episodes_per_epoch,
    tasks_per_episode,
    support_size,
    query_size,
):

    seed_everything(
        seed
    )

    model = (
        MAMLDSCNNSE3C(
            n_channels=22,
            embedding_dim=128,
            n_classes=3,
            n_domains=len(
                TRAIN_SUBJECTS
            ),
        ).to(
            DEVICE
        )
    )

    optimizer = optim.AdamW(
        model.parameters(),
        lr=META_LR,
        weight_decay=WEIGHT_DECAY,
    )

    rng = np.random.default_rng(
        seed
    )

    subject_order = {
        s:i
        for i, s
        in enumerate(
            TRAIN_SUBJECTS
        )
    }

    # Domain loss uses subject ID.
    criterion_cls = nn.CrossEntropyLoss(
        label_smoothing=0.01
    )

    best_state = copy.deepcopy(
        model.state_dict()
    )

    best_meta_loss = np.inf

    history = []

    for epoch in range(
        1,
        epochs + 1,
    ):

        model.train()

        epoch_query_losses = []

        for _ in range(
            episodes_per_epoch
        ):

            optimizer.zero_grad(
                set_to_none=True
            )

            episode_loss = (
                torch.zeros(
                    (),
                    device=DEVICE,
                )
            )

            valid_tasks = 0

            for _task in range(
                tasks_per_episode
            ):

                (
                    Xs,
                    ys,
                    Xq,
                    yq,
                ) = sample_task(
                    X_SOURCE_TRAIN,
                    y_source_train,
                    sub_source_train,
                    rng,
                    support_size=support_size,
                    query_size=query_size,
                )

                Xs_t = torch.from_numpy(
                    Xs
                ).to(
                    DEVICE
                )

                ys_t = torch.from_numpy(
                    ys
                ).long().to(
                    DEVICE
                )

                Xq_t = torch.from_numpy(
                    Xq
                ).to(
                    DEVICE
                )

                yq_t = torch.from_numpy(
                    yq
                ).long().to(
                    DEVICE
                )

                # --------------------------------------------
                # Inner-loop update.
                # --------------------------------------------

                params = dict(
                    model.named_parameters()
                )

                support_logits = (
                    functional_model_call(
                        model,
                        params,
                        Xs_t,
                    )
                )

                support_loss = criterion_cls(
                    support_logits,
                    ys_t,
                )

                grads = torch.autograd.grad(
                    support_loss,
                    tuple(
                        params.values()
                    ),
                    create_graph=False,
                    allow_unused=True,
                )

                fast_params = {}

                for (
                    name,
                    param,
                ), grad in zip(
                    params.items(),
                    grads,
                ):

                    if grad is None:

                        fast_params[
                            name
                        ] = param

                    else:

                        fast_params[
                            name
                        ] = (
                            param
                            - INNER_LR
                            * grad
                        )

                # --------------------------------------------
                # Query loss.
                # --------------------------------------------

                query_logits = (
                    functional_model_call(
                        model,
                        fast_params,
                        Xq_t,
                    )
                )

                query_loss = criterion_cls(
                    query_logits,
                    yq_t,
                )

                episode_loss = (
                    episode_loss
                    + query_loss
                )

                epoch_query_losses.append(
                    float(
                        query_loss.item()
                    )
                )

                valid_tasks += 1

            if valid_tasks == 0:
                continue

            episode_loss = (
                episode_loss
                / valid_tasks
            )

            if torch.isfinite(
                episode_loss
            ):

                episode_loss.backward()

                torch.nn.utils.clip_grad_norm_(
                    model.parameters(),
                    max_norm=3.0,
                )

                optimizer.step()

        mean_meta_loss = (
            float(
                np.mean(
                    epoch_query_losses
                )
            )
            if epoch_query_losses
            else np.inf
        )

        history.append({
            "epoch":
                epoch,
            "meta_query_loss":
                mean_meta_loss,
        })

        if mean_meta_loss < (
            best_meta_loss
        ):

            best_meta_loss = (
                mean_meta_loss
            )

            best_state = (
                copy.deepcopy(
                    model.state_dict()
                )
            )

        print(
            f"    meta epoch {epoch:03d} | "
            f"query_loss={mean_meta_loss:.4f}"
        )

    model.load_state_dict(
        best_state
    )

    return (
        model,
        pd.DataFrame(
            history
        ),
    )

In [13]:
# ============================================================
# CELL 13 — SOURCE VALIDATION HELPERS
# ============================================================

@torch.no_grad()
def predict_model(
    model,
    X,
    batch_size=128,
):

    model.eval()

    ds = TensorDataset(
        torch.from_numpy(
            np.asarray(
                X,
                dtype=np.float32,
            )
        ),
        torch.zeros(
            len(X),
            dtype=torch.long,
        ),
    )

    loader = DataLoader(
        ds,
        batch_size=batch_size,
        shuffle=False,
        num_workers=0,
    )

    outputs = []

    for xb, _ in loader:

        xb = xb.to(
            DEVICE
        )

        logits = model(
            xb
        )

        outputs.append(
            logits.cpu().numpy()
        )

    logits = np.concatenate(
        outputs,
        axis=0,
    )

    logits -= logits.max(
        axis=1,
        keepdims=True,
    )

    probs = np.exp(
        logits
    )

    probs /= (
        probs.sum(
            axis=1,
            keepdims=True,
        )
        + 1e-12
    )

    return probs.astype(
        np.float32
    )


def balanced_source_loader(
    X,
    y,
    subject_array,
    batch_size=64,
):

    y = np.asarray(
        y,
        dtype=np.int64,
    )

    subject_array = np.asarray(
        subject_array
    )

    dataset = TensorDataset(
        torch.from_numpy(
            np.asarray(
                X,
                dtype=np.float32,
            )
        ),
        torch.from_numpy(
            y
        ),
        torch.tensor(
            [
                {
                    s:i
                    for i,s
                    in enumerate(
                        TRAIN_SUBJECTS
                    )
                }[
                    str(s)
                ]
                for s in subject_array
            ],
            dtype=torch.long,
        ),
    )

    counts = np.bincount(
        y,
        minlength=N_CLASSES,
    ).astype(
        np.float64
    )

    inv_class = np.zeros(
        N_CLASSES,
        dtype=np.float64,
    )

    valid = counts > 0

    inv_class[
        valid
    ] = (
        1.0
        / counts[
            valid
        ]
    )

    sample_weights = (
        inv_class[y]
    )

    sampler = WeightedRandomSampler(
        torch.as_tensor(
            sample_weights,
            dtype=torch.double,
        ),
        num_samples=len(y),
        replacement=True,
    )

    return DataLoader(
        dataset,
        batch_size=batch_size,
        sampler=sampler,
        shuffle=False,
        num_workers=0,
    )

In [14]:
# ============================================================
# CELL 14 — DOMAIN-ADVERSARIAL SOURCE FINE-TUNING
# ============================================================

class GradientReversal(
    torch.autograd.Function
):

    @staticmethod
    def forward(
        ctx,
        x,
        lambd,
    ):

        ctx.lambd = (
            float(lambd)
        )

        return x.view_as(
            x
        )

    @staticmethod
    def backward(
        ctx,
        grad_output,
    ):

        return (
            -ctx.lambd
            * grad_output,
            None,
        )


def grl(
    x,
    lambd,
):

    return GradientReversal.apply(
        x,
        lambd,
    )


def domain_train(
    model,
    epochs,
    lr=FINAL_LR,
    domain_lambda=DOMAIN_LAMBDA,
    batch_size=64,
    X_train=None,
    y_train=None,
    subjects_train=None,
    X_val=None,
    y_val=None,
    patience=10,
):

    optimizer = optim.AdamW(
        model.parameters(),
        lr=lr,
        weight_decay=WEIGHT_DECAY,
    )

    scheduler = (
        optim.lr_scheduler.ReduceLROnPlateau(
            optimizer,
            mode="max",
            factor=0.7,
            patience=4,
            min_lr=1e-5,
        )
    )

    criterion_cls = (
        nn.CrossEntropyLoss(
            label_smoothing=0.01
        )
    )

    criterion_domain = (
        nn.CrossEntropyLoss()
    )

    loader = balanced_source_loader(
        X_train,
        y_train,
        subjects_train,
        batch_size=batch_size,
    )

    best_state = copy.deepcopy(
        model.state_dict()
    )

    best_val_bacc = -np.inf
    best_epoch = 1
    wait = 0

    history = []

    for epoch in range(
        1,
        epochs + 1,
    ):

        model.train()

        losses = []
        cls_losses = []
        dom_losses = []

        for xb, yb, db in loader:

            xb = xb.to(
                DEVICE,
                non_blocking=True,
            )

            yb = yb.to(
                DEVICE,
                non_blocking=True,
            )

            db = db.to(
                DEVICE,
                non_blocking=True,
            )

            # Mild source augmentation.
            if torch.rand(1).item() < 0.35:

                xb = (
                    xb
                    * torch.empty(
                        xb.shape[0],
                        1,
                        1,
                        device=DEVICE,
                    ).uniform_(
                        0.94,
                        1.06,
                    )
                )

            if torch.rand(1).item() < 0.20:

                xb = (
                    xb
                    + 0.003
                    * torch.randn_like(
                        xb
                    )
                )

            optimizer.zero_grad(
                set_to_none=True
            )

            logits, features = (
                model(
                    xb,
                    return_features=True,
                )
            )

            loss_cls = (
                criterion_cls(
                    logits,
                    yb,
                )
            )

            domain_logits = (
                model.domain_logits(
                    grl(
                        features,
                        domain_lambda,
                    )
                )
            )

            loss_dom = (
                criterion_domain(
                    domain_logits,
                    db,
                )
            )

            loss = (
                loss_cls
                + loss_dom
            )

            if not torch.isfinite(
                loss
            ):

                continue

            loss.backward()

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                max_norm=3.0,
            )

            optimizer.step()

            losses.append(
                float(loss.item())
            )

            cls_losses.append(
                float(
                    loss_cls.item()
                )
            )

            dom_losses.append(
                float(
                    loss_dom.item()
                )
            )

        P_val = predict_model(
            model,
            X_val,
        )

        pred_val = (
            P_val.argmax(
                axis=1
            )
        )

        val_acc = (
            accuracy_score(
                y_val,
                pred_val,
            )
            * 100.0
        )

        val_bacc = (
            balanced_accuracy_score(
                y_val,
                pred_val,
            )
            * 100.0
        )

        scheduler.step(
            val_bacc
        )

        history.append({
            "epoch":
                epoch,
            "loss":
                np.mean(
                    losses
                ),
            "cls_loss":
                np.mean(
                    cls_losses
                ),
            "domain_loss":
                np.mean(
                    dom_losses
                ),
            "val_acc":
                val_acc,
            "val_bacc":
                val_bacc,
        })

        if val_bacc > (
            best_val_bacc
            + 1e-4
        ):

            best_val_bacc = (
                val_bacc
            )

            best_epoch = epoch

            best_state = (
                copy.deepcopy(
                    model.state_dict()
                )
            )

            wait = 0

        else:

            wait += 1

        if (
            epoch == 1
            or epoch % 5 == 0
        ):

            print(
                f"    epoch {epoch:03d} | "
                f"val={val_acc:5.1f}% | "
                f"bAcc={val_bacc:5.1f}% | "
                f"cls={np.mean(cls_losses):.4f} | "
                f"dom={np.mean(dom_losses):.4f}"
            )

        if wait >= patience:

            print(
                f"    early stop at "
                f"{epoch}; "
                f"best={best_epoch}"
            )

            break

    model.load_state_dict(
        best_state
    )

    return (
        model,
        pd.DataFrame(
            history
        ),
        best_epoch,
        best_val_bacc,
    )

In [16]:
# ============================================================
# CELL 15 — TRAIN TWO SEEDS ON SOURCE TRAIN SUBJECTS
# ============================================================

if SMOKE_MODE:

    meta_epochs = SMOKE_META_EPOCHS
    episodes_per_epoch = (
        SMOKE_EPISODES_PER_EPOCH
    )
    tasks_per_episode = (
        SMOKE_TASKS_PER_EPISODE
    )
    support_size = (
        SMOKE_SUPPORT_PER_TASK
    )
    query_size = (
        SMOKE_QUERY_PER_TASK
    )
    final_epochs = (
        SMOKE_FINAL_EPOCHS
    )
    batch_size = (
        SMOKE_BATCH_SIZE
    )

else:

    meta_epochs = FULL_META_EPOCHS
    episodes_per_epoch = (
        FULL_EPISODES_PER_EPOCH
    )
    tasks_per_episode = (
        FULL_TASKS_PER_EPISODE
    )
    support_size = (
        FULL_SUPPORT_PER_TASK
    )
    query_size = (
        FULL_QUERY_PER_TASK
    )
    final_epochs = (
        FULL_FINAL_EPOCHS
    )
    batch_size = (
        FULL_BATCH_SIZE
    )


seed_models = {}
seed_summaries = {}


for seed in SEEDS:

    print(
        "\n"
        + "=" * 78
    )

    print(
        f"SEED {seed} — FIRST-ORDER MAML SOURCE PRETRAINING"
    )

    print(
        "=" * 78
    )

    (
        maml_model,
        maml_history,
    ) = meta_train_one_seed(
        seed=seed,
        epochs=meta_epochs,
        episodes_per_epoch=episodes_per_epoch,
        tasks_per_episode=tasks_per_episode,
        support_size=support_size,
        query_size=query_size,
    )

    print(
        "\n"
        + "-" * 78
    )

    print(
        f"SEED {seed} — DOMAIN-ADVERSARIAL SOURCE FINE-TUNING"
    )

    print(
        "-" * 78
    )

    (
        final_model,
        final_history,
        best_epoch,
        best_val_bacc,
    ) = domain_train(
        maml_model,
        epochs=final_epochs,
        lr=FINAL_LR,
        domain_lambda=DOMAIN_LAMBDA,
        batch_size=batch_size,
        X_train=X_SOURCE_TRAIN,
        y_train=y_source_train,
        subjects_train=sub_source_train,
        X_val=X_SOURCE_VAL,
        y_val=y_source_val,
        patience=10,
    )

    seed_models[
        seed
    ] = final_model

    seed_summaries[
        seed
    ] = {
        "best_epoch":
            best_epoch,
        "best_val_bacc":
            best_val_bacc,
        "meta_final_loss":
            float(
                maml_history[
                    "meta_query_loss"
                ].iloc[-1]
            ),
    }

    print(
        f"\nSeed {seed}: "
        f"best source validation bAcc = "
        f"{best_val_bacc:.2f}% "
        f"at epoch {best_epoch}"
    )

seed_summary_df = pd.DataFrame(
    [
        {
            "seed":
                seed,
            **seed_summaries[
                seed
            ],
        }
        for seed in SEEDS
    ]
)

print(
    "\nSeed summary:"
)

display(
    seed_summary_df
)


SEED 42 — FIRST-ORDER MAML SOURCE PRETRAINING
    meta epoch 001 | query_loss=1.1026
    meta epoch 002 | query_loss=1.1012
    meta epoch 003 | query_loss=1.1011

------------------------------------------------------------------------------
SEED 42 — DOMAIN-ADVERSARIAL SOURCE FINE-TUNING
------------------------------------------------------------------------------
    epoch 001 | val= 41.5% | bAcc= 41.3% | cls=1.0943 | dom=4.4594
    epoch 005 | val= 41.5% | bAcc= 41.6% | cls=0.9893 | dom=4.0909
    epoch 010 | val= 45.8% | bAcc= 45.9% | cls=0.9134 | dom=4.1647


KeyboardInterrupt: 

In [ ]:
# ============================================================
# CELL 16 — LOAD ALL BCI TARGET SUBJECTS
# ============================================================

def load_target_subject(
    subject,
):

    mask = (
        BCI_META[
            "subject"
        ].astype(str)
        == str(subject)
    )

    idx = (
        BCI_META.loc[
            mask,
            "cache_index",
        ]
        .to_numpy(
            dtype=np.int64
        )
    )

    X = load_indices(
        idx
    )

    y = (
        BCI_META.loc[
            mask,
            "harmonized_class",
        ]
        .map(
            CLASS_TO_ID
        )
        .to_numpy(
            dtype=np.int64
        )
    )

    return (
        X,
        y,
    )


TARGET_RAW = {}
TARGET_STRICT = {}
TARGET_TRANS = {}

for subject in TARGET_SUBJECTS:

    X_raw, y = (
        load_target_subject(
            subject
        )
    )

    TARGET_RAW[
        subject
    ] = (
        X_raw,
        y,
    )

    # Strict: source-normalizer only.
    TARGET_STRICT[
        subject
    ] = (
        SOURCE_NORM.transform(
            X_raw
        ),
        y,
    )

    # Transductive: target unlabeled statistics.
    X_target_norm, _ = (
        target_subject_normalize(
            X_raw
        )
    )

    TARGET_TRANS[
        subject
    ] = (
        X_target_norm,
        y,
    )

    print(
        subject,
        "shape:",
        X_raw.shape,
        "class counts:",
        np.bincount(
            y,
            minlength=3,
        ),
    )

In [ ]:
# ============================================================
# CELL 17 — TWO-SEED STRICT + TRANSDUCTIVE TARGET PREDICTIONS
# ============================================================

def average_seed_predictions(
    models,
    X,
):

    probs = []

    for model in models:

        probs.append(
            predict_model(
                model,
                X,
            )
        )

    return np.mean(
        np.stack(
            probs,
            axis=0,
        ),
        axis=0,
    )


strict_rows = []
trans_rows = []

for subject in TARGET_SUBJECTS:

    Xs, ys = TARGET_STRICT[
        subject
    ]

    Xt, yt = TARGET_TRANS[
        subject
    ]

    P_strict = (
        average_seed_predictions(
            list(
                seed_models.values()
            ),
            Xs,
        )
    )

    P_trans = (
        average_seed_predictions(
            list(
                seed_models.values()
            ),
            Xt,
        )
    )

    pred_strict = (
        P_strict.argmax(
            axis=1
        )
    )

    pred_trans = (
        P_trans.argmax(
            axis=1
        )
    )

    strict_rows.append({
        "subject":
            subject,
        "accuracy":
            accuracy_score(
                ys,
                pred_strict,
            )
            * 100.0,
        "bacc":
            balanced_accuracy_score(
                ys,
                pred_strict,
            )
            * 100.0,
        "kappa":
            cohen_kappa_score(
                ys,
                pred_strict,
            ),
    })

    trans_rows.append({
        "subject":
            subject,
        "accuracy":
            accuracy_score(
                yt,
                pred_trans,
            )
            * 100.0,
        "bacc":
            balanced_accuracy_score(
                yt,
                pred_trans,
            )
            * 100.0,
        "kappa":
            cohen_kappa_score(
                yt,
                pred_trans,
            ),
    })


strict_df = pd.DataFrame(
    strict_rows
)

trans_df = pd.DataFrame(
    trans_rows
)

print(
    "\n"
    + "=" * 78
)

print(
    "CROSS-DATASET EEGMMIDB(109) → BCI-IV-2a(9)"
)

print(
    "=" * 78
)

print(
    "\nSTRICT:"
)

display(
    strict_df
)

print(
    "Mean accuracy:",
    f"{strict_df['accuracy'].mean():.2f}%"
)

print(
    "Mean bAcc:",
    f"{strict_df['bacc'].mean():.2f}%"
)

print(
    "\nTRANSDUCTIVE TARGET NORMALIZATION:"
)

display(
    trans_df
)

print(
    "Mean accuracy:",
    f"{trans_df['accuracy'].mean():.2f}%"
)

print(
    "Mean bAcc:",
    f"{trans_df['bacc'].mean():.2f}%"
)

In [ ]:
# ============================================================
# CELL 18 — FULL TARGET CONFUSION MATRICES
# ============================================================

strict_true_all = []
strict_pred_all = []

trans_true_all = []
trans_pred_all = []

for subject in TARGET_SUBJECTS:

    Xs, ys = TARGET_STRICT[
        subject
    ]

    Xt, yt = TARGET_TRANS[
        subject
    ]

    P_s = (
        average_seed_predictions(
            list(
                seed_models.values()
            ),
            Xs,
        )
    )

    P_t = (
        average_seed_predictions(
            list(
                seed_models.values()
            ),
            Xt,
        )
    )

    strict_true_all.extend(
        ys.tolist()
    )

    strict_pred_all.extend(
        P_s.argmax(
            axis=1
        ).tolist()
    )

    trans_true_all.extend(
        yt.tolist()
    )

    trans_pred_all.extend(
        P_t.argmax(
            axis=1
        ).tolist()
    )


strict_true_all = np.asarray(
    strict_true_all
)

strict_pred_all = np.asarray(
    strict_pred_all
)

trans_true_all = np.asarray(
    trans_true_all
)

trans_pred_all = np.asarray(
    trans_pred_all
)

cm_strict = confusion_matrix(
    strict_true_all,
    strict_pred_all,
    labels=[
        0,
        1,
        2,
    ],
    normalize="true",
)

cm_trans = confusion_matrix(
    trans_true_all,
    trans_pred_all,
    labels=[
        0,
        1,
        2,
    ],
    normalize="true",
)

print(
    "STRICT normalized confusion matrix:"
)

display(
    pd.DataFrame(
        cm_strict,
        index=CLASSES,
        columns=CLASSES,
    ).round(3)
)

print(
    "\nTRANSDUCTIVE normalized confusion matrix:"
)

display(
    pd.DataFrame(
        cm_trans,
        index=CLASSES,
        columns=CLASSES,
    ).round(3)
)

print(
    "\nSTRICT classification report:"
)

print(
    classification_report(
        strict_true_all,
        strict_pred_all,
        labels=[
            0,
            1,
            2,
        ],
        target_names=CLASSES,
        digits=4,
    )
)

print(
    "\nTRANSDUCTIVE classification report:"
)

print(
    classification_report(
        trans_true_all,
        trans_pred_all,
        labels=[
            0,
            1,
            2,
        ],
        target_names=CLASSES,
        digits=4,
    )
)

In [ ]:
# ============================================================
# CELL 19 — SAVE RESULTS / PROTOCOL
# ============================================================

strict_path = (
    RESULT_DIR
    / "strict_eegmmidb109_to_bci9_results.csv"
)

trans_path = (
    RESULT_DIR
    / "transductive_eegmmidb109_to_bci9_results.csv"
)

strict_df.to_csv(
    strict_path,
    index=False,
)

trans_df.to_csv(
    trans_path,
    index=False,
)

protocol = {
    "model":
        "MAML-DSCNN-SE-DA-3C",

    "source_dataset":
        "EEGMMIDB",

    "source_subjects":
        109,

    "target_dataset":
        "BCI-IV-2a",

    "target_subjects":
        9,

    "input_shape":
        [22,640],

    "sampling_rate_hz":
        160,

    "classes":
        CLASSES,

    "encoder":
        "3-branch multi-scale temporal depthwise-separable CNN + depthwise spatial filtering + SE attention + 128-D attentive pooling",

    "meta_learning":
        "first-order MAML-style episodic source training",

    "domain_alignment":
        "gradient-reversal subject-domain classifier during source fine-tuning",

    "ensemble_seeds":
        list(SEEDS),

    "strict_mode":
        {
            "target_data_used_for_training":
                False,
            "target_data_used_for_normalization":
                False,
            "target_labels_used_for_selection":
                False,
        },

    "transductive_mode":
        {
            "target_data_used_for_training":
                False,
            "target_data_used_for_unlabeled_normalization":
                True,
            "target_labels_used_for_selection":
                False,
        },
}

protocol_path = (
    RESULT_DIR
    / "protocol.json"
)

with open(
    protocol_path,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        protocol,
        f,
        indent=2,
    )

print(
    "Saved:",
    strict_path,
)

print(
    "Saved:",
    trans_path,
)

print(
    "Saved:",
    protocol_path,
)

## Final reporting rule

Use the **strict mean over the 9 BCI-IV-2a subjects** as the primary cross-dataset generalization number.

Report the target-statistics result separately as **unsupervised/transductive target normalization**.

Do not use BCI labels to choose:
- architecture,
- seed,
- epoch,
- normalization,
- or any fusion weight.

The target labels are only consumed in the final metric cells.